<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Long-Term Memory with Vector Store_sol </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

# Demo: Long-Term Memory with Vector Store

**Purpose:**  
This notebook demonstrates **Long-Term Memory** using a Vector Store, a production-grade technique that allows AI agents to remember past interactions beyond the current conversation window.

**Scenario:**  
We build a **Financial Research Agent** specialized in **Top AI Service Provider Companies** (Microsoft, Google, Amazon, NVIDIA, IBM).

The agent can answer questions about:
- Company values and mission
- Main projects and profit drivers
- SDG (Sustainable Development Goals) alignment and sustainability efforts
- Financial and strategic insights

It uses **persistent vector store memory** to recall relevant past conversations intelligently.

## 1. Setup

In [ ]:
# Run this first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*" faiss-cpu beautifulsoup4 requests --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.

## 2. Tools – Real Data Scraping for Top AI Companies

In [ ]:
from langchain_core.tools import tool
import requests
from bs4 import BeautifulSoup

@tool
def get_ai_company_info(company: str) -> str:
    """Fetch structured information about top AI service provider companies (Microsoft, Google, Amazon, NVIDIA, IBM)."""
    company = company.lower().strip()
    info_db = {
        "microsoft": "Microsoft is a leader in cloud AI through Azure AI and OpenAI partnership. Main projects: Copilot, Azure OpenAI Service. SDG alignment: SDG 9 (Industry, Innovation), SDG 13 (Climate Action). Profit drivers: Cloud computing and enterprise software.",
        "google": "Google (Alphabet) excels in search AI, Gemini model, and DeepMind. Main projects: Gemini, Google Cloud AI. SDG alignment: SDG 9, SDG 13. Profit drivers: Advertising and cloud services.",
        "amazon": "Amazon leads in e-commerce AI and AWS. Main projects: Amazon Bedrock, SageMaker. SDG alignment: SDG 9, SDG 12 (Responsible Consumption). Profit drivers: AWS cloud and e-commerce.",
        "nvidia": "NVIDIA dominates AI hardware with GPUs. Main projects: CUDA, DGX systems. SDG alignment: SDG 9, SDG 13. Profit drivers: Data center GPUs for AI training.",
        "ibm": "IBM focuses on enterprise AI and Watson. Main projects: Watsonx, Quantum computing. SDG alignment: SDG 9, SDG 17 (Partnerships). Profit drivers: Hybrid cloud and AI consulting."
    }
    return info_db.get(company, f"Information for {company} is not available in this demo. Try Microsoft, Google, Amazon, NVIDIA, or IBM.")

tools = [get_ai_company_info]

## 3. Long-Term Memory with Vector Store

In production systems, we use a **Vector Store** to store past interactions as embeddings.  
This allows the agent to retrieve only the **most relevant** past conversations instead of loading everything.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain.memory import ConversationBufferMemory

embeddings = OpenAIEmbeddings()

# Vector store for long-term memory
vectorstore = FAISS.from_texts(
    ["Initial empty memory"],
    embeddings
)

print("Vector Store Long-Term Memory initialized")

Vector Store Long-Term Memory initialized


## 4. Storing Past Interactions into Vector Store

Every time the agent has a meaningful conversation, we store it as a document in the vector store.

In [ ]:
from datetime import datetime

def store_interaction(query: str, response: str):
    """Store conversation in vector store for long-term retrieval."""
    doc = Document(
        page_content=f"User: {query}\nAssistant: {response}",
        metadata={"timestamp": str(datetime.now())}
    )
    vectorstore.add_documents([doc])
    print(f"Stored interaction in long-term memory: {query[:60]}...")

# Example storage
store_interaction(
    "Tell me about Microsoft's AI initiatives and SDG goals",
    "Microsoft is heavily invested in Azure AI and has committed to multiple SDGs including SDG 9 and SDG 13."
)

Stored interaction in long-term memory: Tell me about Microsoft's AI initiatives and SDG goals...


## 5. Retrieving Relevant Past Interactions

The agent can now retrieve only the **most relevant** past conversations using semantic search.

In [ ]:
def retrieve_relevant_memory(query: str, k: int = 3):
    """Retrieve most relevant past interactions."""
    docs = vectorstore.similarity_search(query, k=k)
    history = "\n\n".join([doc.page_content for doc in docs])
    return history

print("Relevant memory retrieval ready")

Relevant memory retrieval ready


## 6. Full Agent with Long-Term Memory + RAG

We combine:
- Short-term memory (current conversation)
- Long-term vector store memory (past interactions)
- Tool usage (real data fetching)

In [ ]:
# from langchain.agents import create_react_agent, AgentExecutor
# from langchain_openai import ChatOpenAI
# from langsmith import Client

# client = Client()

# llm = ChatOpenAI(model="gpt-4", temperature=0)
# prompt = client.pull_prompt("hwchase17/react",
#     dangerously_pull_public_prompt=True)

# agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

# executor = AgentExecutor(
#     agent=agent,
#     tools=tools,
#     verbose=True,
#     max_iterations=10,
#     handle_parsing_errors=True
# )

# print("Financial Research Agent with Long-Term Vector Memory is ready")

In [ ]:
from langchain.prompts import PromptTemplate

# Prompt
prompt_template = """You are an expert Financial Research Assistant specializing in Top AI Service Provider Companies (Microsoft, Google, Amazon, NVIDIA, IBM).

Your job is to provide clear, factual, and insightful information about their AI projects, company values, main profit drivers, and SDG alignment.

You have access to tools and long-term memory from past interactions.

Use tools when needed and combine information with relevant past conversations.

{tools}

Current conversation history (from long-term memory):
{history}

Use the following format exactly:

Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (repeat if needed)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
{agent_scratchpad}"""

prompt = PromptTemplate.from_template(prompt_template)

In [ ]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langsmith import Client

client = Client()

llm = ChatOpenAI(model="gpt-4", temperature=0)

# Base ReAct prompt (standard)
base_prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)


# We will inject history dynamically, so we keep the standard ReAct prompt
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=12,
    handle_parsing_errors=True
)

print(" Agent created with standard ReAct prompt")

 Agent created with standard ReAct prompt


## 7. Multi-turn Conversation with Long-Term Memory

The agent now remembers past interactions across sessions.

In [ ]:
queries = [
    "Tell me about Microsoft's main AI projects and which SDG goals they follow.",
    "What are NVIDIA's key strengths and sustainability efforts?",
    "Compare the values and long-term vision of Google and Amazon in AI."
]

for q in queries:
    print(f"\n{'='*70}")
    print(f"User: {q}")

    # Retrieve relevant past memory
    past_memory = retrieve_relevant_memory(q)
    if past_memory:
        print(f"(Retrieved from long-term memory)\n{past_memory}\n")

    # Pass memory into the agent via the {history} slot
    response = executor.invoke({
        "input": q,
        "history": past_memory or "No relevant past interactions found."
    })

    # # Run the agent
    # response = executor.invoke({"input": q})

    print("Agent:", response["output"])

    # Store this interaction for future long-term retrieval
    store_interaction(q, response["output"])


User: Tell me about Microsoft's main AI projects and which SDG goals they follow.
(Retrieved from long-term memory)
User: Tell me about Microsoft's AI initiatives and SDG goals
Assistant: Microsoft is heavily invested in Azure AI and has committed to multiple SDGs including SDG 9 and SDG 13.

Initial empty memory



> Entering new AgentExecutor chain...
Thought: I need to fetch the detailed information about Microsoft's AI initiatives and their alignment with SDG goals.
Action: get_ai_company_info
Action Input: MicrosoftMicrosoft is a leader in cloud AI through Azure AI and OpenAI partnership. Main projects: Copilot, Azure OpenAI Service. SDG alignment: SDG 9 (Industry, Innovation), SDG 13 (Climate Action). Profit drivers: Cloud computing and enterprise software.I now have the detailed information about Microsoft's AI initiatives, their main profit drivers, and their alignment with SDG goals.
Final Answer: Microsoft is a leader in the AI space, primarily through its cloud platform, Az